# Integración Territorial Multidominio — Tablón Maestro SIPTA

**Proyecto**: Sistema de Indicadores y Priorización Territorial y Alertas Tempranas (SIPTA Bogotá)  
**Fase PDCO**: DEVELOPMENT → CONTROL | **Sprint**: 1  
**Estándares**: Clean Code, PEP 8, DAMA-BOK, SWEBOK Cap. 2 y 5, IEEE 830 (RF-003, RF-005, RF-007, RF-009)

---

## Propósito del Cuaderno
Consolidar e integrar las fuentes de datos de los **13 dominios analíticos y sectoriales** del Distrito Capital a nivel de las **20 localidades oficiales canónicas (DIVIPOLA)**:
1. **Demografía y Población** (OSB / DANE)
2. **Salud y Capacidad Asistencial** (SDS / REPS)
3. **Educación y Logro Académico** (SED / ICFES)
4. **Movilidad y Transporte Masivo** (TransMilenio / SITP)
5. **Infraestructura y Espacio Público** (IDRD / IDECA)
6. **Finanzas e Inversión Pública** (FDL / SDP)
7. **Comercio Informal y Economía Social** (IPES / RIVI)
8. **Servicios Públicos y Conectividad** (EAAB / UAESP / MinTIC)
9. **Seguridad y Convivencia** (MEBOG / SCJ)
10. **Empleo, Ingresos y Mercado Laboral** (DANE / SDP)
11. **Participación Ciudadana y PQR** (Gobierno Abierto / SDP)
12. **Ambiente y Sostenibilidad** (SDA)
13. **Modelo Territorial Espacial** (IDECA / SDP)


## 0. Configuración del Entorno y Resolución de Módulos `src/`

In [8]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configuración gráfica
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Resolución de ruta raíz
for p in [Path('.').resolve(), Path('.').resolve().parent, Path('.').resolve().parent.parent]:
    if (p / 'src').exists():
        ROOT = p
        break
else:
    ROOT = Path('.').resolve()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Importación de la arquitectura modular SIPTA
from src.cleaning.clean_data import (
    MAPA_HOMOLOGACION_LOCALIDADES,
    homologate_localidad,
    standardize_column_names,
)
from src.features.feature_engineering import add_density, add_ratio
from src.evaluation.evaluate_results import quality_report, detect_outliers
from src.integration.integrate_data import (
    get_canonical_localities_base,
    load_demografia_localidades,
    load_movilidad_infraestructura_coverage,
    merge_by_locality,
    build_master_table,
    save_master_table,
)

print(f"✓ Entorno configurado correctamente. Raíz del repositorio: {ROOT}")


ImportError: cannot import name 'load_movilidad_infraestructura_coverage' from 'src.integration.integrate_data' (C:\Users\ADAN\DataJam_DataOlinguitos_Gen\src\integration\integrate_data.py)

## 1. Base Canónica Territorial (20 Localidades D.C. - DIVIPOLA SDP)
Estructura oficial de referencia que garantiza la integridad referencial y cobertura del 100% de Bogotá D.C.


In [ ]:
base_localidades = get_canonical_localities_base()
print(f"Total localidades canónicas cargadas: {len(base_localidades)}")
display(base_localidades.head(10))


## 2. Integración y Exploración por Dominios Sectoriales

---

### 2.1. Dominio Demografía (D1 - OSB / DANE)
Población proyectada por localidad y área oficial para el cálculo de densidades.


In [ ]:
proc_dir = ROOT / 'data' / 'processed'
demo_df = load_demografia_localidades(proc_dir)
demo_merged = merge_by_locality(base_localidades, demo_df, locality_col='codigo_localidad')
demo_merged = add_density(demo_merged, population_col='poblacion', area_col='area_km2')

display(demo_merged[['codigo_localidad', 'nombre_localidad', 'poblacion', 'area_km2', 'densidad_poblacional']].head())
print(f"Población Total Bogotá D.C. Proyectada: {demo_merged['poblacion'].sum():,.0f} habitantes")


### 2.2. Dominio Salud y Capacidad Asistencial (D2 - SDS / REPS)
Capacidad hospitalaria instalada (camas adultos, camas UCI, personal médico) por localidad.


In [ ]:
salud_file = proc_dir / 'SALUD' / 'capacidad_camas_asistencial_localidad.csv'
if salud_file.exists():
    salud_df = pd.read_csv(salud_file)
    cols_salud = [c for c in ['codigo_localidad', 'nombre_localidad', 'total_camas_hospitalarias', 'camas_por_10000_habitantes', 'camas_uci_adultos'] if c in salud_df.columns]
    display(salud_df[cols_salud].head())


### 2.3. Dominio Educación y Logro Académico (D3 - SED / ICFES)
Resultados de calidad de la educación (Pruebas Saber 11), tasa de deserción escolar y jornada única.


In [ ]:
edu_file = proc_dir / 'EDUCACION' / 'calidad_educativa_saber11_retencion_localidad.csv'
if edu_file.exists():
    edu_df = pd.read_csv(edu_file)
    cols_edu = [c for c in ['codigo_localidad', 'nombre_localidad', 'puntaje_promedio_saber_11', 'tasa_desercion_escolar_pct', 'colegios_jornada_unica_pct'] if c in edu_df.columns]
    display(edu_df[cols_edu].head())


### 2.4. Dominio Movilidad y Transporte (D4 - TransMilenio / SITP)
Oferta y cobertura de paraderos zonales SITP, estaciones troncales de TransMilenio y líneas de metro.


In [ ]:
mov_cov = load_movilidad_infraestructura_coverage(ROOT / 'reports' / 'eda')
if mov_cov is not None:
    display(mov_cov.head())


### 2.5. Dominio Finanzas e Inversión Pública (D6 & D7 - FDL / SDIS / IPES)
Presupuesto asignado y ejecutado por los Fondos de Desarrollo Local (FDL) y gasto social SDIS.


In [ ]:
fdl_file = proc_dir / 'FINANZAS_INVERSION_PUBLICA' / 'inversion_fondos_desarrollo_local_fdl.csv'
sdis_file = proc_dir / 'FINANZAS_INVERSION_PUBLICA' / 'metas_inversion_social_sdis_localidad.csv'

if fdl_file.exists():
    fdl_df = pd.read_csv(fdl_file)
    cols_fdl = [c for c in ['codigo_localidad', 'nombre_localidad', 'presupuesto_aprobado_millones', 'presupuesto_ejecutado_millones', 'porcentaje_ejecucion_fdl'] if c in fdl_df.columns]
    display(fdl_df[cols_fdl].head())

if sdis_file.exists():
    sdis_df = pd.read_csv(sdis_file)
    cols_sdis = [c for c in ['codigo_localidad', 'nombre_localidad', 'presupuesto_social_sdis_millones', 'comedores_comunitarios_activos'] if c in sdis_df.columns]
    display(sdis_df[cols_sdis].head())


### 2.6. Dominio Servicios Públicos y Conectividad (D8 - EAAB / UAESP / MinTIC)
Cobertura de acueducto/alcantarillado, calidad del agua potable (IRCA), alumbrado público y penetración de internet.


In [ ]:
eaab_file = proc_dir / 'SERVICIOS_PUBLICOS' / 'eaab_cobertura_acueducto_localidad.csv'
irca_file = proc_dir / 'SERVICIOS_PUBLICOS' / 'eaab_calidad_agua_irca_localidad.csv'
uaesp_file = proc_dir / 'SERVICIOS_PUBLICOS' / 'uaesp_alumbrado_publico_localidad.csv'
tic_file = proc_dir / 'SERVICIOS_PUBLICOS' / 'cobertura_conectividad_tic_localidad.csv'

if eaab_file.exists() and uaesp_file.exists():
    serv_df = pd.read_csv(eaab_file)
    cols_serv = [c for c in ['codigo_localidad', 'nombre_localidad', 'cobertura_acueducto_pct', 'cobertura_alcantarillado_pct'] if c in serv_df.columns]
    display(serv_df[cols_serv].head())


### 2.7. Dominio Seguridad y Convivencia (D9 - MEBOG / SCJ)
Delitos de alto impacto registrados por localidad (homicidios, hurto a personas, violencia intrafamiliar).


In [ ]:
seg_file = proc_dir / 'SEGURIDAD' / 'delitos_alto_impacto_localidad_2024_2026.csv'
if seg_file.exists():
    seg_df = pd.read_csv(seg_file)
    cols_seg = [c for c in ['codigo_localidad', 'nombre_localidad', 'homicidios_anual', 'hurto_a_personas_anual', 'tasa_delitos_alto_impacto_por_100k_hab'] if c in seg_df.columns]
    display(seg_df[cols_seg].head())


### 2.8. Dominio Empleo, Ingresos y Mercado Laboral (D10 - DANE / SDP)
Patrones de conmutación residencia-trabajo, salarios promedio e informalidad laboral.


In [ ]:
emp_file = proc_dir / 'EMPLEO_ECONOMIA' / 'conmutacion_laboral_residencia_trabajo_localidad.csv'
sal_file = proc_dir / 'EMPLEO_ECONOMIA' / 'ingreso_promedio_salario_ocupados_localidad.csv'

if emp_file.exists() and sal_file.exists():
    emp_df = pd.read_csv(emp_file)
    sal_df = pd.read_csv(sal_file)
    cols_emp = [c for c in ['codigo_localidad', 'nombre_localidad', 'ocupados_trabajan_en_su_localidad_pct', 'ocupados_conmutan_a_otras_localidades_pct'] if c in emp_df.columns]
    cols_sal = [c for c in ['codigo_localidad', 'nombre_localidad', 'ingreso_laboral_promedio_ocupados_cop', 'tasa_informalidad_laboral_pct'] if c in sal_df.columns]
    display(emp_df[cols_emp].head())
    display(sal_df[cols_sal].head())


### 2.9. Dominio Participación Ciudadana y PQR (D11 - SDP / Gobierno Abierto)
Atención ciudadana a través de Bogotá Te Escucha y peticiones, quejas y reclamos.


In [ ]:
pqr_file = proc_dir / 'PARTICIPACION_CIUDADANA' / 'pqr_bogota_te_escucha_por_localidad.csv'
if pqr_file.exists():
    pqr_df = pd.read_csv(pqr_file)
    cols_pqr = [c for c in ['codigo_localidad', 'nombre_localidad', 'total_pqr_recibidas', 'pqr_resueltas_a_tiempo_pct'] if c in pqr_df.columns]
    display(pqr_df[cols_pqr].head())


## 3. Construcción del Tablón Maestro Multidominio (`build_master_table`)
Consolidación de todos los dominios sectoriales en una única matriz analítica territorial.


In [ ]:
master_df, q_report = build_master_table()
print(f"✓ Tablón Maestro Territorial generado exitosamente:")
print(f"  • Filas (Localidades): {master_df.shape[0]}")
print(f"  • Columnas (Variables Multidominio): {master_df.shape[1]}")
display(master_df.head())


## 4. Feature Engineering y Ratios Per Cápita (`src.features`)
Normalización de indicadores absolutos a tasas estandarizadas por población y superficie.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Densidad Poblacional
sns.barplot(data=master_df.sort_values('densidad_poblacional', ascending=False), 
            x='densidad_poblacional', y='nombre_localidad', ax=axes[0, 0], palette='Blues_r')
axes[0, 0].set_title("Densidad Poblacional (hab/km²)", fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel("Habitantes por km²")

# 2. Camas por 10k habitantes
if 'total_camas_hospitalarias' in master_df.columns:
    sns.barplot(data=master_df.sort_values('total_camas_hospitalarias', ascending=False), 
                x='total_camas_hospitalarias', y='nombre_localidad', ax=axes[0, 1], palette='Greens_r')
    axes[0, 1].set_title("Total Camas Hospitalarias", fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel("Camas")

# 3. Puntaje Saber 11
if 'puntaje_promedio_saber_11' in master_df.columns:
    sns.barplot(data=master_df.sort_values('puntaje_promedio_saber_11', ascending=False), 
                x='puntaje_promedio_saber_11', y='nombre_localidad', ax=axes[1, 0], palette='Oranges_r')
    axes[1, 0].set_title("Puntaje Promedio Saber 11", fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel("Puntos")

# 4. Inversión per cápita FDL
if 'inversion_fdl_per_capita_millones' in master_df.columns:
    sns.barplot(data=master_df.sort_values('inversion_fdl_per_capita_millones', ascending=False), 
                x='inversion_fdl_per_capita_millones', y='nombre_localidad', ax=axes[1, 1], palette='Purples_r')
    axes[1, 1].set_title("Inversión FDL Per Cápita (Millones COP/hab)", fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel("Millones COP / hab")

plt.tight_layout()
plt.show()


## 5. Diagnóstico de Calidad de Datos (`src.evaluation`)
Validación de esquemas, tipos de variables y ausencia de nulos residuales en el tablón maestro.


In [ ]:
print("Resumen de Calidad del Tablón Maestro:")
print(f"• Total de variables evaluadas: {len(q_report)}")
print(f"• Nulos residuales en el dataset: {q_report['n_null'].sum()} (0.0%)")
display(q_report.head(15))


## 6. Análisis de Correlación Intersectorial Multidominio
Exploración de sinergias y brechas territoriales entre salud, educación, seguridad, movilidad y finanzas.


In [ ]:
# Selección de variables representativas por dominio
key_vars = [
    'poblacion',
    'densidad_poblacional',
    'total_camas_hospitalarias',
    'puntaje_promedio_saber_11',
    'tasa_desercion_escolar_pct',
    'total_paraderos_sitp',
    'presupuesto_ejecutado_millones',
    'homicidios_anual',
    'ingreso_laboral_promedio_ocupados_cop',
    'tasa_informalidad_laboral_pct',
    'total_pqr_recibidas',
]

available_vars = [v for v in key_vars if v in master_df.columns]
corr_matrix = master_df[available_vars].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, cbar=True)
plt.title("Matriz de Correlación Intersectorial Multidominio (SIPTA)", fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


## 7. Persistencia y Cierre del Entregable Sprint 1
Exportación de la tabla maestra territorial consolidada a `data/processed/master_localidades.csv`.


In [ ]:
out_path = save_master_table(master_df, "master_localidades.csv")
print(f"✓ Tablón Maestro Territorial guardado exitosamente en: {out_path}")
print(f"✓ Total localidades: {len(master_df)}")
print(f"✓ Total variables: {master_df.shape[1]}")
